In [ ]:
# ============================================================
# CMF.tn Financial Reports PDF Scraper
# File: notebooks/scrape_cmf_pdfs.ipynb
# Copy each section between ╔══╗ headers into a separate cell
#
# WHAT THIS DOES:
#   1. Reads all 68 stock tickers from your DB
#   2. For each stock, scrapes all pages on cmf.tn to find PDFs
#   3. Downloads each PDF to an organised folder structure
#   4. Tracks everything in a pdf_metadata table in PostgreSQL
#   5. Skips already-downloaded files safely on re-run
#
# FOLDER STRUCTURE CREATED:
#   data/financials/
#     AMEN BANK/
#       FY_ANNUAL/
#         amen_bank_efd311221.pdf   (FY 2021)
#         amen_bank_efd311222.pdf   (FY 2022)
#       H1_INTERIM/
#         amen_bank_efi300621.pdf   (H1 2021)
#         amen_bank_efi300625.pdf   (H1 2025)
#     BIAT/
#       FY_ANNUAL/ ...
#       H1_INTERIM/ ...
#
# ESTIMATED TIME:  ~1,400 PDFs × 2s = ~47 minutes total
# ESTIMATED SIZE:  2–5 GB (average PDF is 1–4 MB)
# ============================================================
 

In [5]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports and setup                              ║
# ╚══════════════════════════════════════════════════════════╝
 
import requests
from bs4 import BeautifulSoup
import psycopg2
import psycopg2.extras
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, date
from pathlib import Path
from urllib.parse import quote_plus
import time
import os
import re
 
load_dotenv()
 
# ── DB connection ─────────────────────────────────────────
def get_conn():
    return psycopg2.connect(
        host=os.getenv('DB_HOST'),
        port=int(os.getenv('DB_PORT', 5432)),
        dbname=os.getenv('DB_NAME'),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD')
    )
 
# ── Base folder for all PDFs ─────────────────────────────
# Change this path if your project is in a different location
PDF_BASE_DIR = Path(r'C:\Users\Negza\Desktop\projects\pfe\bvmt_project\data\financials')
 
# Create the base folder if it does not exist
PDF_BASE_DIR.mkdir(parents=True, exist_ok=True)
print(f"PDF folder: {PDF_BASE_DIR}")
print(f"Exists: {PDF_BASE_DIR.exists()}")
 
# ── HTTP Session ──────────────────────────────────────────
# CMF.tn is a Drupal site — pages render server-side, no JS needed
# requests + BeautifulSoup works directly
SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) '
        'Gecko/20100101 Firefox/124.0'
    ),
    'Accept':          'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'fr-TN,fr;q=0.9,en-US;q=0.8',
    'Accept-Encoding': 'gzip, deflate',
    'Referer':         'https://www.cmf.tn/',
    'Connection':      'keep-alive',
})
 
# Warm up session with homepage
try:
    resp = SESSION.get('https://www.cmf.tn/', timeout=20)
    print(f"CMF homepage: HTTP {resp.status_code}")
except Exception as e:
    print(f"Warning: could not reach CMF homepage: {e}")
 
print("Imports OK")
 

PDF folder: C:\Users\Negza\Desktop\projects\pfe\bvmt_project\data\financials
Exists: True
CMF homepage: HTTP 200
Imports OK


In [6]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Fetch all CMF names + smart matching           ║
# ╚══════════════════════════════════════════════════════════╝
#
# STRATEGY (3 steps):
#
# Step 1 — One HTTP request to the CMF search page fetches
#   the complete <select> dropdown with all 644 legal names.
#   The option VALUE is exactly what goes in the URL parameter.
#
# Step 2 — Auto-match each DB ticker to the CMF legal name
#   that contains the most matching keywords.
#   Works perfectly for ~58 of 68 stocks automatically.
#
# Step 3 — Manual overrides fix the ~10 stocks where
#   auto-match picked the wrong company:
#     BIAT matched 'ASSURANCES BIAT' (insurance) not the bank
#     STB  matched 'FCP STB EVOLUTIF' (fund) not the bank
#     UIB  matched 'FCP PERSONNEL UIB' (fund) not the bank
#     TUNIS RE matched 'AIR LIQUIDE TUNISIE' (completely wrong)
#     TUNISAIR matched 'FCP BIAT CEA PNT TUNISAIR' (fund)
#     SAH  matched 'AMS' (different company entirely)
#     AIR LIQUDE TSIE had no match (now manually set)
#     BH   had no match (now manually set)
 
def fetch_cmf_dropdown() -> list:
    """
    Fetch the CMF search page and extract all option values
    from the company select dropdown. One HTTP request gets
    all 644 company names — no scraping loop needed.
    """
    url = ('https://www.cmf.tn/'
           '?q=consultation-des-tats-financier-des-soci-t-s-faisant-ape')
 
    resp = SESSION.get(url, timeout=30)
    soup = BeautifulSoup(resp.text, 'html.parser')
 
    select = soup.find('select', attrs={'name': 'field_societesape_value'})
    if not select:
        raise RuntimeError("Could not find company dropdown on CMF page")
 
    names = []
    for option in select.find_all('option'):
        val = option.get('value', '').strip()
        # Skip blank, 'Any', and filter options
        if val and 'Any' not in val and val != '' and not val.startswith('- '):
            names.append(val)
 
    return names
 
 
def auto_match(db_tickers: list, cmf_names: list) -> dict:
    """
    For each DB ticker, find the CMF legal name that contains
    the most matching keywords.
 
    Scoring rules:
    - Each ticker word (>2 chars, not a stopword) that appears
      in the CMF name scores +1
    - Exact match scores +10 (highest priority)
    - FCP/FCPR/SICAV prefix names are penalised -5
      (these are investment funds, not operating companies)
    """
    STOPWORDS = {'THE','AND','DES','LES','PAR','SUR','DU','DE',
                 'ET','SA','STE','STÉ','GROUPE','HOLDING'}
    FUND_PREFIXES = ('FCP ', 'FCPR ', 'SICAV ', 'SICAR ')
 
    cmf_upper = [(name, name.upper()) for name in cmf_names]
    name_map  = {}
 
    for ticker in db_tickers:
        ticker_up    = ticker.upper()
        ticker_words = [w for w in ticker_up.split()
                        if len(w) > 2 and w not in STOPWORDS]
 
        best_name  = ticker  # fallback: use ticker itself
        best_score = 0
 
        for cmf_name, cmf_up in cmf_upper:
            # Exact match — highest priority
            if cmf_up == ticker_up:
                best_name  = cmf_name
                best_score = 999
                break
 
            # Count matching words
            score = sum(1 for w in ticker_words if w in cmf_up)
 
            # Penalise investment fund names
            if any(cmf_name.upper().startswith(p) for p in FUND_PREFIXES):
                score -= 5
 
            if score > best_score:
                best_score = score
                best_name  = cmf_name
 
        name_map[ticker] = best_name
 
    return name_map
 
 
# ── MANUAL OVERRIDES ──────────────────────────────────────────────────────
# These 10 stocks need a manual fix because:
#   - auto-match picked a fund (FCP/FCPR) with a similar name
#   - auto-match picked a subsidiary/related company
#   - the stock has no matching keywords in its CMF legal name
#
# How to find the correct value for any other stock:
#   Visit CMF, type the stock name in the search bar, note the
#   exact text in the dropdown, then add it here.
 
MANUAL_OVERRIDES = {
    # ── Stocks where auto-match picked a fund instead of the company ──────
    'BIAT':             'BIAT',
    # 'ASSURANCES BIAT' is the insurance arm — we want the bank itself
    # CMF lists 'BIAT' separately (exact match for the bank holding)
 
    'STB':              'STB',
    # 'FCP STB EVOLUTIF' is a fund managed by STB — we want the bank
    # CMF lists 'STB' separately
 
    'UIB':              'UIB',
    # 'FCP PERSONNEL UIB EPARGNE ACTIONS' is a staff savings fund
    # CMF lists 'UIB' separately
 
    'TUNISAIR':         'TUNISAIR',
    # 'FCP BIAT CEA PNT TUNISAIR' is a fund — we want the airline
    # CMF lists 'TUNISAIR' separately
 
    # ── Stocks where auto-match picked the completely wrong company ────────
    'TUNIS RE':         'TUNIS RE',
    # Auto-match wrongly gave 'AIR LIQUIDE TUNISIE' (matched 'TUNISI' substring)
    # CMF lists 'TUNIS RE' separately
 
    'SAH':              'GROUPE SAH',
    # Auto-match gave 'AMS' (both have short names causing collision)
    # SAH on BVMT = Société Al Houda, listed as 'GROUPE SAH' on CMF
 
    # ── Stocks with no auto-match (short or ambiguous names) ─────────────
    'BH':               'BANQUE DE L\'HABITAT',
    # BH = Banque de l'Habitat — CMF uses full legal name
    # 'BH' alone matched nothing (too short, 2 chars)
 
    'AIR LIQUDE TSIE':  'AIR LIQUIDE TUNISIE',
    # Note: your DB has a typo 'LIQUDE' — CMF uses 'LIQUIDE TUNISIE'
    # Auto-match failed because 'LIQUDE' ≠ 'LIQUIDE'
 
    # ── Stocks that CMF may list under a different name ───────────────────
    'ASTREE':           'ASTREE',
    # CMF lists it as 'ASTREE' (exact match — auto-match should work,
    # but adding here as safety override)
 
    'BH ASSURANCE':     'BH ASSURANCE',
    # CMF likely lists as 'BH ASSURANCE' — override to be safe
    # If this returns 0 PDFs, try 'ASSURANCE BH' or 'BH Assurances'
 
    # ── These 4 were not in the CMF dropdown output ───────────────────────
    # They may exist under a different name or may not be APE companies
    'PLAC. TSIE-SICAF': 'PLACEMENT DE TUNISIE SICAF',
    'TUNINVEST-SICAR':  'TUNINVEST-SICAR',
    'BTE (ADP)':        'BTE',
    'ELBENE INDUSTRIE': 'ELBENE',
    'SITEX':            'SITEX',
    'STEQ':             'STEQ',
    'SIPHAT':           'SIPHAT',
    'ALKIMIA':          'ALKIMIA',
}
 
 
# ── Build the final mapping ───────────────────────────────────────────────
print("Step 1 — Fetching CMF dropdown (one request)...")
cmf_names_list = fetch_cmf_dropdown()
print(f"  {len(cmf_names_list)} company names fetched")
 
print("Step 2 — Auto-matching tickers to CMF legal names...")
conn = get_conn()
with conn.cursor() as cur:
    cur.execute('SELECT ticker FROM company_metadata ORDER BY ticker')
    db_tickers = [row[0] for row in cur.fetchall()]
conn.close()
 
CMF_NAME_MAP = auto_match(db_tickers, cmf_names_list)
 
print("Step 3 — Applying manual overrides...")
override_count = 0
for ticker, correct_name in MANUAL_OVERRIDES.items():
    if ticker in CMF_NAME_MAP:
        old = CMF_NAME_MAP[ticker]
        CMF_NAME_MAP[ticker] = correct_name
        if old != correct_name:
            print(f"  Override: {ticker:<28} {old} → {correct_name}")
            override_count += 1
 
print(f"  {override_count} overrides applied")
print()
 
# ── Print full verified mapping ───────────────────────────────────────────
print(f"{'DB Ticker':<30} {'CMF Legal Name'}")
print("-" * 78)
for ticker in sorted(CMF_NAME_MAP.keys()):
    cmf = CMF_NAME_MAP[ticker]
    marker = ' *' if cmf != ticker else ''
    print(f"  {ticker:<30} {cmf}{marker}")
 
print()
print(f"Total: {len(CMF_NAME_MAP)} stocks mapped")
 
 

Step 1 — Fetching CMF dropdown (one request)...
  643 company names fetched
Step 2 — Auto-matching tickers to CMF legal names...
Step 3 — Applying manual overrides...
  Override: BIAT                         ASSURANCES BIAT → BIAT
  Override: STB                          GROUPE Sté. TUNISIENNE DE BANQUE - STB - → STB
  Override: UIB                          GROUPE UNION INTERNATIONALE DE BANQUES - UIB - → UIB
  Override: TUNISAIR                     GROUPE Sté. TUNISIENNE DE L'AIR - TUNISAIR - → TUNISAIR
  Override: TUNIS RE                     AIR LIQUIDE TUNISIE → TUNIS RE
  Override: SAH                          ATELIERS MECANIQUES DU SAHEL - AMS - → GROUPE SAH
  Override: BH                           BH → BANQUE DE L'HABITAT
  7 overrides applied

DB Ticker                      CMF Legal Name
------------------------------------------------------------------------------
  ADWYA                          Sté. ADWYA S.A *
  AMEN BANK                      AMEN BANK
  AMS               

In [3]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Create pdf_metadata table in DB                ║
# ╚══════════════════════════════════════════════════════════╝
#
# This table tracks every PDF we find and download.
# It serves three purposes:
#   1. Skip already-downloaded files on re-run (resume capability)
#   2. Feed into financial_statements later — we know which PDF
#      corresponds to which stock, period, and report type
#   3. Academic audit trail — you can prove exactly what data
#      was used to train your models
 
CREATE_SQL = '''
CREATE TABLE IF NOT EXISTS pdf_metadata (
    id              SERIAL PRIMARY KEY,
    ticker          VARCHAR(50)  NOT NULL,
    isin_code       VARCHAR(20),
 
    -- Report identification
    filename        VARCHAR(200) NOT NULL,
    report_type     VARCHAR(20)  NOT NULL,  -- 'FY_ANNUAL' or 'H1_INTERIM'
    period          VARCHAR(20)  NOT NULL,  -- 'FY 2021' or 'H1 2025'
    period_end_date DATE,                   -- 2021-12-31 or 2025-06-30
    fiscal_year     INTEGER,                -- 2021, 2022, etc.
 
    -- Source information
    cmf_url         VARCHAR(500) NOT NULL,
    local_path      VARCHAR(500),           -- full path on disk after download
 
    -- Status tracking
    download_status VARCHAR(20)  DEFAULT 'pending',
    -- pending | downloaded | failed | skipped
    file_size_bytes BIGINT,
    error_message   VARCHAR(500),
 
    -- Timestamps
    found_at        TIMESTAMP DEFAULT NOW(),
    downloaded_at   TIMESTAMP,
 
    CONSTRAINT uq_pdf UNIQUE (ticker, filename)
);
 
CREATE INDEX IF NOT EXISTS idx_pdf_ticker  ON pdf_metadata (ticker);
CREATE INDEX IF NOT EXISTS idx_pdf_type    ON pdf_metadata (report_type);
CREATE INDEX IF NOT EXISTS idx_pdf_year    ON pdf_metadata (fiscal_year);
CREATE INDEX IF NOT EXISTS idx_pdf_status  ON pdf_metadata (download_status);
'''
 
conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute(CREATE_SQL)
    conn.commit()
    print("pdf_metadata table created (or already exists)")
except Exception as e:
    conn.rollback()
    print(f"Error: {e}")
finally:
    conn.close()
 
 

pdf_metadata table created (or already exists)


In [8]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — Core parsing functions                         ║
# ╚══════════════════════════════════════════════════════════╝
#
# CMF HTML STRUCTURE (confirmed from real page):
#
# Each report is a <div class="views-row ..."> containing:
#
#   <div class="group-first">
#     <span class="date-display-single" content="2022-04-08T00:00:00+01:00">
#     <div class="field-name-field-societesape">AMEN BANK</div>
#   </div>
#   <div class="group-second">
#     <div class="field-name-field-exercice">2021</div>      ← fiscal year
#   </div>
#   <div class="group-third">
#     <div class="field-name-field-p-riode">
#       Etats financiers au 31/12                            ← period type
#     </div>
#   </div>
#   <div class="group-fourth">
#     <div class="field-name-field-pdf-cf">
#       <a href="https://www.cmf.tn/sites/default/files/...pdf">
#     </div>
#   </div>
#
# KEY INSIGHT — two type codes in filenames:
#   efd = états financiers définitifs  → FY_ANNUAL   (31/12)
#   efi = états financiers intérimaires → H1_INTERIM  (30/06)
 
def decode_filename(filename: str) -> dict:
    name = filename.replace('.pdf', '').replace('.PDF', '')

    # Find type code — the part that starts with 'ef'
    type_code = None
    for part in name.split('_'):
        if part.lower().startswith('ef'):
            type_code = part.lower()
            break

    # STRATEGY 1 — standard format: last 6 chars = DDMMYY
    # e.g. amen_bank_efd311221 → 311221 → 2021-12-31
    date_str = name[-6:]
    period_end_date = None
    fiscal_year = None
    try:
        day, month, year = int(date_str[0:2]), int(date_str[2:4]), int('20' + date_str[4:6])
        if 1 <= day <= 31 and 1 <= month <= 12 and 2010 <= year <= 2030:
            period_end_date = date(year, month, day)
            fiscal_year = year
    except (ValueError, TypeError):
        pass

    # STRATEGY 2 — dashed date: efi_au_30-06-21
    if not period_end_date:
        m = re.search(r'(\d{2})-(\d{2})-(\d{2})', name)
        if m:
            try:
                day, month, year = int(m.group(1)), int(m.group(2)), int('20' + m.group(3))
                period_end_date = date(year, month, day)
                fiscal_year = year
            except (ValueError, TypeError):
                pass

    # STRATEGY 3 — year-first format: efi_2020_amen_bank or efd_2019_amen_bank
    if not period_end_date:
        m = re.search(r'(\d{4})', name)
        if m:
            year = int(m.group(1))
            if 2010 <= year <= 2030:
                fiscal_year = year
                if type_code and 'd' in type_code and 'i' not in type_code:
                    period_end_date = date(year, 12, 31)
                else:
                    period_end_date = date(year, 6, 30)

    # Determine report type — efd=annual, efi=interim
    if type_code and 'd' in type_code and 'i' not in type_code:
        report_type = 'FY_ANNUAL'
        period = f'FY {fiscal_year}' if fiscal_year else 'UNKNOWN'
    else:
        report_type = 'H1_INTERIM'
        period = f'H1 {fiscal_year}' if fiscal_year else 'UNKNOWN'

    return {
        'report_type':     report_type,
        'period':          period,
        'period_end_date': str(period_end_date) if period_end_date else None,
        'fiscal_year':     fiscal_year,
    }
 
def scrape_cmf_page(ticker: str, cmf_name: str, page: int) -> list:
    """
    Fetch one page of CMF results for a stock and extract PDF metadata.
 
    Returns a list of dicts, one per PDF found on this page.
    Returns empty list if the page has no results (= last page passed).
 
    URL pattern:
      Page 0 (first):  ?q=...&field_societesape_value=AMEN+BANK&page=0
      Page 1 (second): ?q=...&field_societesape_value=AMEN+BANK&page=1
    Note: page=0 and page= (no page param) both give page 1.
    We always include &page=N for consistency.
    """
    base_url = (
        'https://www.cmf.tn/'
        '?q=consultation-des-tats-financier-des-soci-t-s-faisant-ape'
        f'&field_societesape_value={quote_plus(cmf_name)}'
        f'&page={page}'
    )
 
    pdfs = []
 
    try:
        resp = SESSION.get(base_url, timeout=20)
 
        if resp.status_code == 403:
            time.sleep(30)
            resp = SESSION.get(base_url, timeout=20)
 
        if resp.status_code != 200:
            return []
 
        soup = BeautifulSoup(resp.text, 'html.parser')
 
        # Find all report rows — each has class "views-row"
        rows = soup.find_all('div', class_=re.compile(r'views-row'))
 
        for row in rows:
            # ── Extract PDF URL ─────────────────────────────────
            # The PDF link is inside group-fourth > field-name-field-pdf-cf
            pdf_div = row.find('div', class_=re.compile(r'field-name-field-pdf-cf'))
            if not pdf_div:
                continue
 
            # Find the <a> tag with an href pointing to a PDF file
            pdf_link = None
            for a in pdf_div.find_all('a'):
                href = a.get('href', '')
                if '.pdf' in href.lower() or '.PDF' in href:
                    pdf_link = href.strip()
                    break
 
            if not pdf_link:
                continue
 
            # Make sure it's an absolute URL
            if pdf_link.startswith('//'):
                pdf_link = 'https:' + pdf_link
            elif pdf_link.startswith('/'):
                pdf_link = 'https://www.cmf.tn' + pdf_link
 
            # ── Extract fiscal year from page ───────────────────
            year_div = row.find('div', class_=re.compile(r'field-name-field-exercice'))
            fiscal_year_text = ''
            if year_div:
                year_item = year_div.find('div', class_='field-item')
                if year_item:
                    fiscal_year_text = year_item.get_text(strip=True)
 
            # ── Extract period type text ────────────────────────
            period_div = row.find('div', class_=re.compile(r'field-name-field-p-riode'))
            period_text = ''
            if period_div:
                period_item = period_div.find('div', class_='field-item')
                if period_item:
                    period_text = period_item.get_text(strip=True)
 
            # ── Decode from filename ────────────────────────────
            filename = pdf_link.split('/')[-1]
            decoded  = decode_filename(filename)
 
            # Use fiscal year from HTML if filename decode failed
            if not decoded['fiscal_year'] and fiscal_year_text.isdigit():
                decoded['fiscal_year'] = int(fiscal_year_text)
                decoded['period']      = f"FY {fiscal_year_text}" if '31/12' in period_text else f"H1 {fiscal_year_text}"
 
            # ── Filter by year range ────────────────────────────
            # Only keep 2016–2026 to match our price data
            if decoded['fiscal_year'] and not (2016 <= decoded['fiscal_year'] <= 2026):
                continue
 
            pdfs.append({
                'ticker':          ticker,
                'filename':        filename,
                'cmf_url':         pdf_link,
                'report_type':     decoded['report_type'],
                'period':          decoded['period'],
                'period_end_date': decoded['period_end_date'],
                'fiscal_year':     decoded['fiscal_year'],
            })
 
    except Exception as e:
        print(f"    Error on page {page} for {ticker}: {e}")
 
    return pdfs
 
 
# Quick test
print("Testing CMF parser with AMEN BANK page 0...")
test_pdfs = scrape_cmf_page('AMEN BANK', 'AMEN BANK', page=0)
print(f"Found {len(test_pdfs)} PDFs on page 0")
for p in test_pdfs[:3]:
    print(f"  {p['filename']:<45} {p['report_type']:<14} {p['period']}")
 

Testing CMF parser with AMEN BANK page 0...
Found 19 PDFs on page 0
  amen_bank_efi300625.pdf                       H1_INTERIM     H1 2025
  amen_bank_efd311224.pdf                       FY_ANNUAL      FY 2024
  amen_bank_efi300624.pdf                       H1_INTERIM     H1 2024


In [10]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — Scrape all pages for one stock                 ║
# ╚══════════════════════════════════════════════════════════╝
#
# HOW STOP CONDITION WORKS:
# CMF uses zero-indexed pagination: page=0, page=1, page=2...
# When you request a page beyond the last one, CMF returns a
# valid HTML page but with ZERO views-row divs (no results).
# We detect this: if scrape_cmf_page() returns [] → stop.
# Safety cap at max_pages=20 (no stock should have 20+ pages).
 
def scrape_all_pages_cmf(ticker: str, cmf_name: str, max_pages: int = 20) -> list:
    """
    Scrape ALL pages of CMF results for one stock.
    Returns combined list of all PDF metadata dicts found.
    """
    all_pdfs = []
    page     = 0
 
    while page < max_pages:
        pdfs = scrape_cmf_page(ticker, cmf_name, page)
 
        if not pdfs:
            # Empty page = we have passed the last page → stop
            break
 
        all_pdfs.extend(pdfs)
        page += 1
        time.sleep(2)  # polite pause between pages
 
    return all_pdfs
 
 
def save_pdf_metadata_to_db(pdfs: list, isin_map: dict) -> int:
    """
    Save discovered PDF metadata to pdf_metadata table.
    Returns count of NEW records inserted.
    """
    if not pdfs:
        return 0
 
    conn = get_conn()
    inserted = 0
    try:
        with conn.cursor() as cur:
            for p in pdfs:
                isin = isin_map.get(p['ticker'])
                try:
                    cur.execute('''
                        INSERT INTO pdf_metadata
                            (ticker, isin_code, filename, report_type, period,
                             period_end_date, fiscal_year, cmf_url, download_status)
                        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, 'pending')
                        ON CONFLICT (ticker, filename) DO NOTHING
                    ''', (
                        p['ticker'], isin, p['filename'],
                        p['report_type'], p['period'],
                        p['period_end_date'], p['fiscal_year'],
                        p['cmf_url']
                    ))
                    if cur.rowcount == 1:
                        inserted += 1
                except Exception as e:
                    print(f"    DB error for {p['filename']}: {e}")
        conn.commit()
    finally:
        conn.close()
 
    return inserted
 
 
# ── Load ISIN map from DB ─────────────────────────────────
conn = get_conn()
with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
    cur.execute('SELECT ticker, isin_code FROM company_metadata')
    isin_map = {row['ticker']: row['isin_code'] for row in cur.fetchall()}
conn.close()
print(f"ISIN map: {len(isin_map)} stocks loaded")
 
# ── Test with AMEN BANK ───────────────────────────────────
print("\nTesting full page scan for AMEN BANK...")
test_all = scrape_all_pages_cmf('AMEN BANK', 'AMEN BANK')
print(f"Total PDFs found: {len(test_all)}")
for p in test_all:
    print(f"  {p['filename']:<45} {p['report_type']:<14} {p['period']}")
 
saved = save_pdf_metadata_to_db(test_all, isin_map)
print(f"\nSaved {saved} new records to pdf_metadata table")
 
 

ISIN map: 70 stocks loaded

Testing full page scan for AMEN BANK...
Total PDFs found: 19
  amen_bank_efi300625.pdf                       H1_INTERIM     H1 2025
  amen_bank_efd311224.pdf                       FY_ANNUAL      FY 2024
  amen_bank_efi300624.pdf                       H1_INTERIM     H1 2024
  amen_bank_efd311223.pdf                       FY_ANNUAL      FY 2023
  amen_bank_efi300623.pdf                       H1_INTERIM     H1 2023
  amen_bank_efd311222.pdf                       FY_ANNUAL      FY 2022
  amen_bank_efi300622.pdf                       H1_INTERIM     H1 2022
  amen_bank_efd311221.pdf                       FY_ANNUAL      FY 2021
  amen_bank_efi_au_30-06-21.pdf                 H1_INTERIM     H1 2021
  amen_bank_efd311220.pdf                       FY_ANNUAL      FY 2020
  efi_2020_amen_bank.pdf                        H1_INTERIM     H1 2020
  efd_2019_amen_bank.pdf                        FY_ANNUAL      FY 2019
  amen_bank_efi300619.pdf                       H1_INTERIM 

In [11]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Scan all stocks with retry logic               ║
# ╚══════════════════════════════════════════════════════════╝
#
# CHANGES FROM PREVIOUS VERSION:
# - Uses CMF_NAME_MAP (auto-discovered + corrected names)
# - Retry on timeout: 3 attempts with 30s wait between each
# - Delay: 4s between stocks (was 2s — was causing timeouts)
# - Cooldown: 60s every 15 stocks (was every 20)
# - Clears old 'pending' records before re-scanning
 
 
def fetch_cmf_page_safe(ticker: str, cmf_name: str, page: int,
                         max_retries: int = 3) -> list:
    """
    Fetch one CMF page with automatic retry on timeout.
    Waits 30 seconds between retries.
    Returns [] if all retries fail.
    """
    for attempt in range(1, max_retries + 1):
        try:
            return scrape_cmf_page(ticker, cmf_name, page)
 
        except Exception as e:
            err_str = str(e).lower()
            is_timeout = 'timeout' in err_str or 'timed out' in err_str
 
            if is_timeout and attempt < max_retries:
                print(f"\n    [Timeout attempt {attempt}/{max_retries} — waiting 30s]",
                      end="", flush=True)
                time.sleep(30)
            else:
                if is_timeout:
                    print(f"\n    [All {max_retries} retries failed — skipping]")
                else:
                    print(f"\n    [Error: {str(e)[:60]}]")
                return []
 
    return []
 
 
def scrape_all_pages_safe(ticker: str, cmf_name: str,
                           max_pages: int = 20) -> list:
    """Scrape all pages for one stock using the retry-safe fetcher."""
    all_pdfs = []
    page     = 0
 
    while page < max_pages:
        pdfs = fetch_cmf_page_safe(ticker, cmf_name, page)
 
        if not pdfs:
            break  # empty = last page reached or all retries failed
 
        all_pdfs.extend(pdfs)
        page += 1
        time.sleep(3)  # pause between pages of the same stock
 
    return all_pdfs
 
 
def scan_all_stocks_final():
    """
    Full scan of all stocks using corrected CMF names.
    Clears previously pending records first for a clean run.
    """
    # ── Clear old pending records from the failed first scan ─────────────
    conn = get_conn()
    with conn.cursor() as cur:
        cur.execute("DELETE FROM pdf_metadata WHERE download_status = 'pending'")
        deleted = cur.rowcount
    conn.commit()
    conn.close()
    if deleted > 0:
        print(f"Cleared {deleted} stale pending records from previous scan")
        print()
 
    print("=" * 65)
    print("CMF Metadata Scan — final run")
    print("=" * 65)
    print(f"Stocks:         {len(CMF_NAME_MAP)}")
    print(f"Delay:          4s between stocks, 3s between pages")
    print(f"Cooldown:       60s every 15 stocks")
    print(f"Retries:        3 attempts per page on timeout")
    print(f"Estimated time: ~15 minutes")
    print("=" * 65)
    print()
 
    grand_total = 0
    zero_stocks = []
 
    for i, (ticker, cmf_name) in enumerate(CMF_NAME_MAP.items(), 1):
        print(f"[{i:2d}/{len(CMF_NAME_MAP)}] {ticker:<30}", end=" ", flush=True)
 
        pdfs  = scrape_all_pages_safe(ticker, cmf_name)
        saved = save_pdf_metadata_to_db(pdfs, isin_map)
        grand_total += len(pdfs)
 
        if not pdfs:
            print(f"→ 0 PDFs  (tried: '{cmf_name}')")
            zero_stocks.append((ticker, cmf_name))
        else:
            fy = sum(1 for p in pdfs if p['report_type'] == 'FY_ANNUAL')
            h1 = sum(1 for p in pdfs if p['report_type'] == 'H1_INTERIM')
            print(f"→ {len(pdfs):3d} PDFs  (FY={fy} H1={h1})  saved={saved}")
 
        time.sleep(4)
 
        if i % 15 == 0:
            print(f"  [60s cooldown after {i} stocks...]")
            time.sleep(60)
 
    # ── Final report ──────────────────────────────────────────────────────
    print()
    print("=" * 65)
    print("SCAN COMPLETE")
    print(f"  Total PDFs discovered: {grand_total:,}")
    print(f"  Stocks with PDFs:      {len(CMF_NAME_MAP) - len(zero_stocks)}")
    print(f"  Stocks with 0 PDFs:    {len(zero_stocks)}")
 
    if zero_stocks:
        print()
        print("  Zero-result stocks:")
        print("  (These are listed on BVMT but may not file with CMF APE,")
        print("   or CMF uses a completely different name)")
        for ticker, tried in zero_stocks:
            print(f"    {ticker:<30} tried: '{tried}'")
        print()
        print("  For each: visit cmf.tn, open the search bar,")
        print("  type the company name, and note what autocomplete shows.")
        print("  Then call: rescrape_single_stock(ticker, correct_name)")
 
    return grand_total, zero_stocks
 
 
# Run the final scan
total_found, zero_list = scan_all_stocks_final()
 

Cleared 19 stale pending records from previous scan

CMF Metadata Scan — final run
Stocks:         70
Delay:          4s between stocks, 3s between pages
Cooldown:       60s every 15 stocks
Retries:        3 attempts per page on timeout
Estimated time: ~15 minutes

[ 1/70] ADWYA                          →  14 PDFs  (FY=7 H1=7)  saved=14
[ 2/70] AMEN BANK                      →  19 PDFs  (FY=9 H1=10)  saved=19
[ 3/70] AMS                            →   6 PDFs  (FY=4 H1=2)  saved=6
[ 4/70] ARTES                          →  19 PDFs  (FY=9 H1=10)  saved=19
[ 5/70] ASSAD                          →  10 PDFs  (FY=0 H1=10)  saved=10
[ 6/70] ASSU MAGHREBIA VIE             →  12 PDFs  (FY=7 H1=5)  saved=12
[ 7/70] ASSUR MAGHREBIA                →  11 PDFs  (FY=4 H1=7)  saved=11
[ 8/70] ATB                            →  19 PDFs  (FY=9 H1=10)  saved=19
[ 9/70] ATELIER MEUBLE INT             → 0 PDFs  (tried: 'GROUPE Sté. ATELIER DU MEUBLE intérieurs')
[10/70] ATL                            →  19 P

In [12]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 10 — Fix zero-result stocks (targeted re-scrape)   ║
# ╚══════════════════════════════════════════════════════════╝
#
# Instead of re-running the full 15-minute scan, this cell
# only re-scrapes the 14 stocks that returned 0 PDFs.
# The 720 already-discovered PDFs are untouched.
#
# WHAT WE NOW KNOW ABOUT EACH ZERO-RESULT STOCK:
#
# CONFIRMED WRONG NAMES (now corrected):
#   BIAT            -> 'BANQUE INTERNATIONALE ARABE DE TUNISIE - BIAT -'
#   STB             -> 'Sté. TUNISIENNE DE BANQUE - STB -'
#   UIB             -> 'UNION INTERNATIONALE DE BANQUES - UIB -'
#   SAH             -> 'Sté. D\'ARTICLES HYGIENIQUES - SAH -'
#   ENNAKL AUTOMOBILES -> 'Sté. ENNAKL AUTOMOBILES'
#   BH              -> 'BANQUE DE L\'HABITAT'  (apostrophe encoding issue)
#   BT              -> 'BANQUE DE TUNISIE'
#   ATELIER MEUBLE INT -> 'GROUPE Sté. ATELIER DU MEUBLE intérieurs'
#   TUNIS RE        -> 'TUNIS-RE'  (hyphen variant)
#
# CONFIRMED NON-FILERS (skip — no CMF APE financial statements):
#   MONOPRIX        -> not in CMF APE list (private retail chain)
#   UADH            -> not in CMF APE list (recent listing)
#   TUNISAIR        -> state-owned, may not file H1/FY with CMF APE
#   GIF-FILTER      -> small cap, likely not APE registered
#   BH LEASING      -> same company as MODERN LEASING (already scraped)
 
ZERO_FIXES = {
    'BIAT':              'BANQUE INTERNATIONALE ARABE DE TUNISIE - BIAT -',
    'STB':               'Sté. TUNISIENNE DE BANQUE - STB -',
    'UIB':               'UNION INTERNATIONALE DE BANQUES - UIB -',
    'SAH':               "Sté. D'ARTICLES HYGIENIQUES - SAH -",
    'ENNAKL AUTOMOBILES':'Sté. ENNAKL AUTOMOBILES',
    'BH':                "BANQUE DE L'HABITAT",
    'BT':                'BANQUE DE TUNISIE',
    'ATELIER MEUBLE INT':'GROUPE Sté. ATELIER DU MEUBLE intérieurs',
    'TUNIS RE':          'TUNIS-RE',
}
 
# These are confirmed non-filers — skip entirely
NON_FILERS = {'MONOPRIX', 'UADH', 'TUNISAIR', 'GIF-FILTER', 'BH LEASING'}
 
print("Targeted re-scrape for zero-result stocks")
print(f"Fixing:   {len(ZERO_FIXES)} stocks with corrected names")
print(f"Skipping: {len(NON_FILERS)} confirmed non-filers")
print()
 
total_new = 0
still_zero = []
 
for ticker, correct_name in ZERO_FIXES.items():
    print(f"  {ticker:<28} '{correct_name}'", end=" ", flush=True)
 
    pdfs = scrape_all_pages_safe(ticker, correct_name)
 
    if not pdfs:
        print("→ still 0")
        still_zero.append((ticker, correct_name))
    else:
        saved = save_pdf_metadata_to_db(pdfs, isin_map)
        total_new += len(pdfs)
        fy = sum(1 for p in pdfs if p['report_type'] == 'FY_ANNUAL')
        h1 = sum(1 for p in pdfs if p['report_type'] == 'H1_INTERIM')
        print(f"→ {len(pdfs)} PDFs  (FY={fy} H1={h1})  saved={saved}")
 
    time.sleep(4)
 
print()
print(f"New PDFs discovered: {total_new}")
print(f"Previously found:    720")
print(f"New total:           {720 + total_new}")
 
if still_zero:
    print()
    print("Still 0 after correction — these are genuine non-filers:")
    for t, n in still_zero:
        print(f"  {t:<28} tried: '{n}'")
 
print()
print("Confirmed non-filers (skipped):")
for t in NON_FILERS:
    print(f"  {t}")
 

Targeted re-scrape for zero-result stocks
Fixing:   9 stocks with corrected names
Skipping: 5 confirmed non-filers

  BIAT                         'BANQUE INTERNATIONALE ARABE DE TUNISIE - BIAT -'     Error on page 0 for BIAT: HTTPSConnectionPool(host='www.cmf.tn', port=443): Read timed out. (read timeout=20)
→ still 0
  STB                          'Sté. TUNISIENNE DE BANQUE - STB -' → 19 PDFs  (FY=9 H1=10)  saved=19
  UIB                          'UNION INTERNATIONALE DE BANQUES - UIB -' → 19 PDFs  (FY=9 H1=10)  saved=19
  SAH                          'Sté. D'ARTICLES HYGIENIQUES - SAH -' → 20 PDFs  (FY=10 H1=10)  saved=20
  ENNAKL AUTOMOBILES           'Sté. ENNAKL AUTOMOBILES' → 19 PDFs  (FY=6 H1=13)  saved=19
  BH                           'BANQUE DE L'HABITAT' → still 0
  BT                           'BANQUE DE TUNISIE' → still 0
  ATELIER MEUBLE INT           'GROUPE Sté. ATELIER DU MEUBLE intérieurs' → still 0
  TUNIS RE                     'TUNIS-RE' → still 0

New PDFs discov

In [13]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 12 — Fix ATELIER MEUBLE INT + retry BIAT           ║
# ╚══════════════════════════════════════════════════════════╝

# Fix 1 — ATELIER MEUBLE INT
# Previous attempt used 'GROUPE Sté. ATELIER DU MEUBLE intérieurs'
# Correct name is:        'Sté. ATELIER DU MEUBLE intérieurs'  (no GROUPE)

print("Fixing ATELIER MEUBLE INT...", end=" ", flush=True)
pdfs = scrape_all_pages_safe('ATELIER MEUBLE INT', 'Sté. ATELIER DU MEUBLE intérieurs')
if pdfs:
    saved = save_pdf_metadata_to_db(pdfs, isin_map)
    fy = sum(1 for p in pdfs if p['report_type'] == 'FY_ANNUAL')
    h1 = sum(1 for p in pdfs if p['report_type'] == 'H1_INTERIM')
    print(f"→ {len(pdfs)} PDFs  (FY={fy} H1={h1})  saved={saved}")
else:
    print("→ still 0")

time.sleep(5)

# Fix 2 — BIAT was a timeout, name is correct, just retry
print("Retrying BIAT...", end=" ", flush=True)
pdfs = scrape_all_pages_safe('BIAT', 'BANQUE INTERNATIONALE ARABE DE TUNISIE - BIAT -')
if pdfs:
    saved = save_pdf_metadata_to_db(pdfs, isin_map)
    fy = sum(1 for p in pdfs if p['report_type'] == 'FY_ANNUAL')
    h1 = sum(1 for p in pdfs if p['report_type'] == 'H1_INTERIM')
    print(f"→ {len(pdfs)} PDFs  (FY={fy} H1={h1})  saved={saved}")
else:
    print("→ timeout again — run this cell once more in 5 minutes")

# Summary
conn = get_conn()
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM pdf_metadata WHERE download_status='pending'")
    total = cur.fetchone()[0]
conn.close()
print(f"\nTotal PDFs ready to download: {total}")

Fixing ATELIER MEUBLE INT...     Error on page 0 for ATELIER MEUBLE INT: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
→ still 0
Retrying BIAT... → 19 PDFs  (FY=9 H1=10)  saved=19

Total PDFs ready to download: 816


In [14]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 11 — Final verification query                      ║
# ╚══════════════════════════════════════════════════════════╝
 
conn = get_conn()
 
df = pd.read_sql('''
    SELECT
        ticker,
        COUNT(*) FILTER (WHERE report_type = 'FY_ANNUAL')            AS fy,
        COUNT(*) FILTER (WHERE report_type = 'H1_INTERIM')           AS h1,
        COUNT(*)                                                       AS total,
        MIN(fiscal_year)                                               AS from_yr,
        MAX(fiscal_year)                                               AS to_yr
    FROM pdf_metadata
    WHERE download_status = 'pending'
    GROUP BY ticker
    ORDER BY total DESC, ticker
''', conn)
 
total_pdfs = df['total'].sum()
print(f"Total PDFs ready to download: {total_pdfs:,}")
print(f"Stocks covered: {len(df)}")
print()
print(df.to_string(index=False))
 
conn.close()
 
print()
print("Run Cell 7 (the download cell) to start downloading all PDFs.")
print(f"Estimated time: ~{total_pdfs * 2 // 60} minutes")
print(f"Estimated size: ~{total_pdfs * 2:.0f} MB – {total_pdfs * 4:.0f} MB")

Total PDFs ready to download: 816
Stocks covered: 61

            ticker  fy  h1  total  from_yr  to_yr
     NEW BODY LINE   8  12     20     2016   2025
               SAH  10  10     20     2016   2025
          SOTRAPIL  10  10     20     2016   2025
         AMEN BANK   9  10     19     2016   2025
             ARTES   9  10     19     2016   2025
               ATB   9  10     19     2016   2025
               ATL   9  10     19     2016   2025
     ATTIJARI BANK   9  10     19     2016   2025
  ATTIJARI LEASING   9  10     19     2016   2025
              BIAT   9  10     19     2016   2025
               BNA   9  10     19     2016   2025
   CARTHAGE CEMENT   9  10     19     2016   2025
           CELLCOM   9  10     19     2016   2025
               CIL   9  10     19     2016   2025
CIMENTS DE BIZERTE   9  10     19     2016   2025
ENNAKL AUTOMOBILES   6  13     19     2016   2025
          ESSOUKNA   8  11     19     2016   2025
    HANNIBAL LEASE   9  10     19     2016   2

C:\Users\Negza\AppData\Local\Temp\ipykernel_5008\2635936019.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql('''


In [15]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Download all PDFs                              ║
# ╚══════════════════════════════════════════════════════════╝
#
# FOLDER STRUCTURE:
#   data/financials/
#     AMEN BANK/
#       FY_ANNUAL/
#         amen_bank_efd311221.pdf
#       H1_INTERIM/
#         amen_bank_efi300625.pdf
#
# RESUME CAPABILITY:
#   If a PDF already exists on disk AND its size matches → skip.
#   This means you can stop and restart the download safely.
#   The pdf_metadata table tracks status: pending/downloaded/failed.
#
# ESTIMATED TIME: ~1,400 PDFs × 2s = ~47 minutes
# Do not close Jupyter or sleep your computer during this.
 
def get_local_path(ticker: str, report_type: str, filename: str) -> Path:
    """
    Build the local file path for a PDF.
    Creates parent directories automatically.
    """
    # Sanitise ticker name for use as folder name
    # (remove characters that are invalid in Windows folder names)
    safe_ticker = re.sub(r'[<>:"/\\|?*]', '_', ticker)
 
    folder = PDF_BASE_DIR / safe_ticker / report_type
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename
 
 
def download_one_pdf(row: dict) -> tuple:
    """
    Download a single PDF from CMF.
 
    Returns (status, file_size_bytes, error_message)
      status: 'downloaded' | 'skipped' | 'failed'
 
    Skips if file already exists and has the correct size.
    """
    local_path = get_local_path(row['ticker'], row['report_type'], row['filename'])
 
    # ── Already downloaded? ─────────────────────────────────
    # Check by existence AND non-zero size (a 0-byte file = failed download)
    if local_path.exists() and local_path.stat().st_size > 1000:
        return 'skipped', local_path.stat().st_size, None
 
    try:
        resp = SESSION.get(row['cmf_url'], timeout=60, stream=True)
 
        if resp.status_code == 404:
            return 'failed', 0, 'HTTP 404 — file not found on CMF server'
 
        if resp.status_code != 200:
            return 'failed', 0, f'HTTP {resp.status_code}'
 
        # Write PDF in chunks (stream=True avoids loading full file into RAM)
        total_bytes = 0
        with open(local_path, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    total_bytes += len(chunk)
 
        # Verify we got a real PDF (PDFs start with %PDF)
        if total_bytes < 1000:
            local_path.unlink()  # delete the broken file
            return 'failed', 0, f'File too small ({total_bytes} bytes) — not a valid PDF'
 
        return 'downloaded', total_bytes, None
 
    except Exception as e:
        # Clean up partial file if it exists
        if local_path.exists():
            local_path.unlink()
        return 'failed', 0, str(e)
 
 
def update_pdf_status(pdf_id: int, status: str, local_path: str,
                      file_size: int, error_msg: str):
    """Update the download status in pdf_metadata table."""
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('''
                UPDATE pdf_metadata
                SET download_status = %s,
                    local_path      = %s,
                    file_size_bytes = %s,
                    error_message   = %s,
                    downloaded_at   = CASE WHEN %s = 'downloaded' THEN NOW() ELSE NULL END
                WHERE id = %s
            ''', (status, local_path, file_size, error_msg, status, pdf_id))
        conn.commit()
    finally:
        conn.close()
 
 
def run_full_download():
    """
    Download all PDFs that are currently 'pending' in pdf_metadata.
    Can be re-run safely — already downloaded files are skipped.
    """
 
    # Load all pending PDFs from DB
    conn = get_conn()
    df = pd.read_sql('''
        SELECT id, ticker, filename, report_type, cmf_url
        FROM pdf_metadata
        WHERE download_status IN ('pending', 'failed')
        ORDER BY ticker, fiscal_year, report_type
    ''', conn)
    conn.close()
 
    total     = len(df)
    downloaded = 0
    skipped    = 0
    failed     = 0
    total_mb   = 0
 
    print("=" * 65)
    print(f"CMF PDF Download — {total} files to process")
    print(f"Saving to: {PDF_BASE_DIR}")
    print("=" * 65)
 
    for i, row in df.iterrows():
        seq = downloaded + skipped + failed + 1
        print(f"[{seq:4d}/{total}] {row['ticker']:<25} {row['filename']:<40}", end=" ", flush=True)
 
        status, size_bytes, error = download_one_pdf(row)
        size_mb = size_bytes / (1024 * 1024)
        total_mb += size_mb
 
        local_path_str = str(get_local_path(row['ticker'], row['report_type'], row['filename']))
        update_pdf_status(row['id'], status, local_path_str, size_bytes, error)
 
        if status == 'downloaded':
            downloaded += 1
            print(f"✓ {size_mb:.1f} MB")
        elif status == 'skipped':
            skipped += 1
            print(f"↷ already exists ({size_mb:.1f} MB)")
        else:
            failed += 1
            print(f"✗ {error}")
 
        # Polite pause between downloads
        time.sleep(1.5)
 
        # Longer pause every 50 downloads to avoid rate limiting
        if seq % 50 == 0:
            print(f"  [30s cooldown after {seq} files — {total_mb:.0f} MB so far]")
            time.sleep(30)
 
    print()
    print("=" * 65)
    print("DOWNLOAD COMPLETE")
    print(f"  Downloaded:  {downloaded:,} new PDFs")
    print(f"  Skipped:     {skipped:,} already existed")
    print(f"  Failed:      {failed:,}")
    print(f"  Total size:  {total_mb:.0f} MB ({total_mb/1024:.1f} GB)")
    print("=" * 65)
 
    if failed > 0:
        print()
        print("Failed downloads — these can be retried:")
        conn = get_conn()
        df_fail = pd.read_sql('''
            SELECT ticker, filename, error_message
            FROM pdf_metadata WHERE download_status = 'failed'
            ORDER BY ticker
        ''', conn)
        conn.close()
        print(df_fail.to_string(index=False))
 
    return downloaded
 
 
# Run the download
# Make sure Cell 6 (scan) has finished before running this.
total_dl = run_full_download()
 
 

CMF PDF Download — 816 files to process
Saving to: C:\Users\Negza\Desktop\projects\pfe\bvmt_project\data\financials
[   1/816] ADWYA                     adwya_efd311216.pdf                      

C:\Users\Negza\AppData\Local\Temp\ipykernel_5008\1151013593.py:110: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql('''


✗ ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
[   2/816] ADWYA                     adwya_efi300616.pdf                      ✓ 0.1 MB
[   3/816] ADWYA                     adwya_efd311217.pdf                      ✓ 0.1 MB
[   4/816] ADWYA                     adwya_efi300617.pdf                      ✓ 0.1 MB
[   5/816] ADWYA                     adwya_efd311218.pdf                      ✓ 0.1 MB
[   6/816] ADWYA                     adwya_efi300618.pdf                      ✓ 0.1 MB
[   7/816] ADWYA                     adwya_efd_31122019.pdf                   ✓ 0.8 MB
[   8/816] ADWYA                     adwya_efi300619.pdf                      ✗ HTTPSConnectionPool(host='www.cmf.tn', port=443): Read timed out.
[   9/816] ADWYA                     adwya_efd311220.pdf                      ✓ 0.1 MB
[  10/816] ADWYA                     adwya_efi300620.pdf                      ✓ 1.0 MB
[  11/816] ADWYA                     adwya_efd311221.pdf       

C:\Users\Negza\AppData\Local\Temp\ipykernel_5008\1151013593.py:171: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_fail = pd.read_sql('''


In [16]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL — Retry 2 failed ADWYA PDFs                        ║
# ╚══════════════════════════════════════════════════════════╝

print("Waiting 30s before retrying...")
time.sleep(30)

conn = get_conn()
df_fail = pd.read_sql(
    "SELECT id, ticker, filename, report_type, cmf_url "
    "FROM pdf_metadata WHERE download_status = 'failed'",
    conn
)
conn.close()

print(f"Retrying {len(df_fail)} failed PDFs...")
for _, row in df_fail.iterrows():
    print(f"  {row['filename']:<45}", end=" ", flush=True)
    status, size, error = download_one_pdf(row)
    path_str = str(get_local_path(row['ticker'], row['report_type'], row['filename']))
    update_pdf_status(row['id'], status, path_str, size, error)
    icon = '✓' if status == 'downloaded' else '✗'
    print(f"{icon}  {status}  {error or ''}")
    time.sleep(5)

print("\nDone.")

Waiting 30s before retrying...
Retrying 2 failed PDFs...
  adwya_efi300619.pdf                           

C:\Users\Negza\AppData\Local\Temp\ipykernel_5008\1099974710.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_fail = pd.read_sql(


✗  failed  ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
  adwya_efd311216.pdf                           ✓  downloaded  

Done.


In [17]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8 — Verify and report                              ║
# ╚══════════════════════════════════════════════════════════╝
 
conn = get_conn()
 
# Summary by stock and report type
df_summary = pd.read_sql('''
    SELECT
        ticker,
        COUNT(*) FILTER (WHERE report_type = 'FY_ANNUAL')   AS fy_count,
        COUNT(*) FILTER (WHERE report_type = 'H1_INTERIM')  AS h1_count,
        COUNT(*) FILTER (WHERE download_status = 'downloaded') AS downloaded,
        COUNT(*) FILTER (WHERE download_status = 'failed')    AS failed,
        MIN(fiscal_year) AS earliest_year,
        MAX(fiscal_year) AS latest_year,
        ROUND(SUM(file_size_bytes) / 1048576.0, 1) AS total_mb
    FROM pdf_metadata
    GROUP BY ticker
    ORDER BY downloaded DESC, ticker
''', conn)
 
print(f"Stocks with PDFs: {len(df_summary)}")
print(f"Total PDFs in metadata: {df_summary['downloaded'].sum() + df_summary['failed'].sum()}")
print(f"Total downloaded: {df_summary['downloaded'].sum()}")
print(f"Total MB on disk: {df_summary['total_mb'].sum():.0f} MB")
print()
print(df_summary.to_string(index=False))
 
# Year coverage check
df_years = pd.read_sql('''
    SELECT fiscal_year, report_type, COUNT(*) AS pdfs
    FROM pdf_metadata
    WHERE download_status = 'downloaded'
    GROUP BY fiscal_year, report_type
    ORDER BY fiscal_year, report_type
''', conn)
 
print()
print("Coverage by year:")
print(df_years.to_string(index=False))
 
conn.close()
 
 

Stocks with PDFs: 61
Total PDFs in metadata: 816
Total downloaded: 815
Total MB on disk: 906 MB

            ticker  fy_count  h1_count  downloaded  failed  earliest_year  latest_year  total_mb
     NEW BODY LINE         8        12          20       0           2016         2025       7.3
               SAH        10        10          20       0           2016         2025      12.0
          SOTRAPIL        10        10          20       0           2016         2025     127.3
         AMEN BANK         9        10          19       0           2016         2025      29.4
             ARTES         9        10          19       0           2016         2025       8.5
               ATB         9        10          19       0           2016         2025      18.6
               ATL         9        10          19       0           2016         2025      26.3
     ATTIJARI BANK         9        10          19       0           2016         2025      22.0
  ATTIJARI LEASING         9  

C:\Users\Negza\AppData\Local\Temp\ipykernel_5008\719570311.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_summary = pd.read_sql('''
C:\Users\Negza\AppData\Local\Temp\ipykernel_5008\719570311.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_years = pd.read_sql('''


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 9 — Fix zero-result stocks (manual name lookup)    ║
# ╚══════════════════════════════════════════════════════════╝
#
# Some stocks will return 0 PDFs because CMF uses a slightly
# different name. This cell helps you find the correct name.
#
# HOW TO USE:
#   1. Run Cell 6 first and note which stocks show 0 PDFs
#   2. For each one, visit CMF manually and search for the stock
#   3. Copy the exact name from the search results
#   4. Update CMF_NAMES in Cell 2 with the correct name
#   5. Re-run Cells 6 and 7 for just those stocks (see below)
 
def rescrape_single_stock(ticker: str, correct_cmf_name: str):
    """
    Re-scrape and re-download a single stock after fixing its CMF name.
    Use this for stocks that showed 0 PDFs in the main scan.
    """
    print(f"Re-scanning {ticker} with CMF name: '{correct_cmf_name}'")
 
    pdfs  = scrape_all_pages_cmf(ticker, correct_cmf_name)
    saved = save_pdf_metadata_to_db(pdfs, isin_map)
    print(f"  Found {len(pdfs)} PDFs, saved {saved} new records")
 
    if pdfs:
        print(f"  Downloading...")
        conn = get_conn()
        df = pd.read_sql(
            "SELECT id, ticker, filename, report_type, cmf_url FROM pdf_metadata "
            "WHERE ticker=%s AND download_status='pending'",
            conn, params=(ticker,)
        )
        conn.close()
 
        for _, row in df.iterrows():
            status, size, error = download_one_pdf(row)
            path_str = str(get_local_path(row['ticker'], row['report_type'], row['filename']))
            update_pdf_status(row['id'], status, path_str, size, error)
            icon = '✓' if status == 'downloaded' else '↷' if status == 'skipped' else '✗'
            print(f"    {icon} {row['filename']}")
 
# EXAMPLE USAGE (uncomment and run after identifying the correct name):
# rescrape_single_stock('GIF-FILTER', 'GIF FILTER INTERNATIONAL')
# rescrape_single_stock('ATL', 'ASSURANCES TAKAFUL LINA')
 